# Build reproducible samples

In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

DATA_ROOT = Path("../../data")
SOURCE_DIR = DATA_ROOT
SAMPLE_DIR = DATA_ROOT / "sample_dataset"
JSON_DIR = DATA_ROOT / "sample_json"
GROUND_TRUTH_DIR = DATA_ROOT / "ground_truth"

RANDOM_STATE = 42
ROWS_PER_MONTH = 10
GROUND_TRUTH_RECORDS = 5 # sample random records evenly across the date range

## Load and prepare data

In [ ]:
def load_data(base_dir: Path) -> pd.DataFrame:
    if not base_dir.is_dir():
        raise FileNotFoundError(f"Source directory does not exist: {base_dir}")

    records = []
    for json_path in sorted(base_dir.rglob("*.json")):
        try:
            with json_path.open("r", encoding="utf-8") as file:
                record = json.load(file)

            if not isinstance(record, dict):
                raise ValueError("JSON root is not an object")

            record["_source_folder"] = json_path.parent.name
            record["_file_name"] = json_path.name
            records.append(record)

        except (json.JSONDecodeError, OSError, ValueError) as error:
            print(f"Could not read {json_path}: {error}")

    return pd.DataFrame(records)


def prepare_data(data: pd.DataFrame) -> pd.DataFrame:
    prepared = data.copy()
    prepared["Prüfdatum"] = pd.to_datetime(prepared["Prüfdatum"], dayfirst=True, errors="coerce")
    prepared["Jahr"] = prepared["Prüfdatum"].dt.year.astype("Int64")
    prepared["Prüfnummer"] = pd.to_numeric(prepared["Prüfnummer"], errors="coerce")
    
    return prepared


In [ ]:
df = prepare_data(load_data(SOURCE_DIR))
print(f"Loaded {len(df):,} data records.")

## Sampling functions

In [ ]:
def sample_by_month(
    data: pd.DataFrame,
    rows_per_month: int = 10,
    random_state: int = 42,
    date_col: str = "Prüfdatum",
) -> pd.DataFrame:
    if rows_per_month < 1:
        raise ValueError("rows_per_month must be at least 1")
    if date_col not in data:
        raise KeyError(f"Missing date column: {date_col}")

    working = data.copy()
    working["_sample_date"] = pd.to_datetime(
        working[date_col], dayfirst=True, errors="coerce"
    )
    working = working.loc[working["_sample_date"].notna()].copy()
    working["_sample_year"] = working["_sample_date"].dt.year
    working["_sample_month"] = working["_sample_date"].dt.month
    working["_sample_order"] = np.random.default_rng(random_state).random(
        len(working)
    )

    sampled = (
        working.sort_values(
            ["_sample_year", "_sample_month", "_sample_order"]
        )
        .groupby(["_sample_year", "_sample_month"], sort=True)
        .head(rows_per_month)
        .sort_values(["_sample_year", "_sample_month", "_sample_order"])
        .drop(columns=["_sample_date", "_sample_year", "_sample_month", "_sample_order"])
        .reset_index(drop=True)
    )
    return sampled


def sample_ground_truth(
    data: pd.DataFrame,
    number_of_records: int = 5,
    random_state: int = 42,
    date_col: str = "Prüfdatum",
) -> pd.DataFrame:
    if number_of_records < 1:
        raise ValueError("number_of_records must be at least 1")
    if date_col not in data:
        raise KeyError(f"Missing date column: {date_col}")

    working = data.copy()
    working["_sample_date"] = pd.to_datetime(
        working[date_col], dayfirst=True, errors="coerce"
    )
    working = working.loc[working["_sample_date"].notna()].copy()
    working = working.sort_values("_sample_date")

    if len(working) < number_of_records:
        raise ValueError(
            f"At least {number_of_records} dated records are required"
        )

    boundaries = np.linspace(
        0, len(working), number_of_records + 1, dtype=int
    )
    rng = np.random.default_rng(random_state)
    selected_positions = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        selected_positions.append(start + int(rng.integers(end - start)))

    sampled = (
        working.iloc[selected_positions]
        .drop(columns=["_sample_date"])
        .reset_index(drop=True)
    )

    if len(sampled) != number_of_records:
        raise RuntimeError(
            f"Could not select exactly {number_of_records} ground truth records"
        )
    return sampled


## Create both samples

In [39]:
sample_df = sample_by_month(
    df, rows_per_month=ROWS_PER_MONTH, random_state=RANDOM_STATE
)
print(f"Sample: {len(sample_df):,} records")
sample_df.head()

Sample: 2,824 records


,Titel_und_Signatur,Bemerkung,Prüfnummer,Prüfdatum,Antragsteller,Produktion,Land,Filmart,Stumm-/Tonfilm,Länge (in Meter),...,movie_subtitle,produced_by,director,cinematography,set_design,cast_list,excerpt_note,length_details,examination_certificate,Jahr
0,R 9346-I/27347\nSicher und bequem\n1920 - 1945,NaN,44585.0,1917-01-27,"Tolirag, Ton- und Lichtbildreklame AG, Berlin","Tolirag, Ton- und Lichtbildreklame AG, Berlin",Deutschland,Werbefilm,Stummfilm,47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1917
1,R 9346-I/27293\nVon Kanada in den deutschen Wa...,NaN,44503.0,1917-01-16,"Paul Lieberenz Filmproduktion, Berlin","Paul Lieberenz Filmproduktion, Berlin",Deutschland,Kulturfilm,Stummfilm,414,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1917
2,R 9346-I/27363\nVon Katzen und Grosskatzen\n19...,NaN,44615.0,1917-02-16,"Paul Lieberenz-Filmproduktion, Berlin","Paul Lieberenz-Filmproduktion, Berlin",Deutschland,Kulturfilm,Stummfilm,371,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1917
3,R 9346-I/174\nAfrikanisches Abenteuer\n1920 - ...,NaN,915.0,1920-01-09,"Werbefilm GmbH, Berlin","Werbefilm GmbH, Berlin",Deutschland,Kulturfilm,Stummfilm,50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1920
4,R 9346-I/114\nDie Eisbärenjagd\n1920 - 1945,NaN,626.0,1920-01-01,John Hagenbeck-Film,John Hagenbeck-Film,Deutschland,Zeichentrickfilm,Stummfilm,203,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1920


In [47]:
ground_truth_df = sample_ground_truth(
    df, number_of_records=GROUND_TRUTH_RECORDS, random_state=RANDOM_STATE
)
print(f"Ground truth: {len(ground_truth_df):,} records")
ground_truth_df

Ground truth: 5 records


,Titel_und_Signatur,Bemerkung,Prüfnummer,Prüfdatum,Antragsteller,Produktion,Land,Filmart,Stumm-/Tonfilm,Länge (in Meter),...,movie_subtitle,produced_by,director,cinematography,set_design,cast_list,excerpt_note,length_details,examination_certificate,Jahr
0,R 9346-I/109\nRudi's stilles Stündchen.\n1920 ...,NaN,584.0,1920-10-15,"Terra-Film-AG, Berlin","Terra-Film-AG, Berlin",Deutschland,Spielfilm,Stummfilm,625,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1920
1,R 9346-I/13991\nDer deutsche Dauerschwimmer Ot...,NaN,20202.0,1928-09-22,"Otto Kemmerich, Husum","Otto Kemmerich, Husum",Deutschland,Kulturfilm,Stummfilm,321,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1928
2,R 9346-I/22050\nArmer kleiner Held\n1920 - 1945,"Nach Kürzung 2028,20 m",33466.0,1933-04-01,"Europa-Filmverleih AG, Berlin","Vandal & Delac, Paris",Frankreich,Spielfilm,Tonfilm,2093,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1933
3,R 9346-I/27959\nSchmetterlinge\n1920 - 1945,NaN,45894.0,1937-08-06,"Tobis-Melofilm GmbH, Berlin","Tobis-Melofilm GmbH, Berlin",Deutschland,Kulturfilm,Tonfilm,271,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1937
4,R 9346-I/34828\nEin Geheimnis mit Carl Napp\n1...,NaN,57104.0,1942-04-22,"Epoche-Color-Film AG, Berlin","Epoche-Color-Film AG, Berlin",Deutschland,Werbefilm,Tonfilm,64,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1942


## Copy source folders

In [ ]:
def copy_folders(
    data: pd.DataFrame,
    source_dir: Path,
    target_dir: Path,
    folder_col: str = "_source_folder",
    overwrite: bool = False,
) -> dict[str, int]:
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Source directory does not exist: {source_dir}")
    if folder_col not in data:
        raise KeyError(f"Missing folder column: {folder_col}")

    target_dir.mkdir(parents=True, exist_ok=True)
    summary = {"copied": 0, "skipped": 0, "missing": 0}

    folders = sorted(data[folder_col].dropna().astype(str).unique())
    for folder in folders:
        source_folder = source_dir / folder
        target_folder = target_dir / folder

        if not source_folder.is_dir():
            summary["missing"] += 1
            print(f"Source folder not found: {source_folder}")
            continue
        if target_folder.exists() and not overwrite:
            summary["skipped"] += 1
            continue

        shutil.copytree(source_folder, target_folder, dirs_exist_ok=overwrite)
        summary["copied"] += 1

    return summary


In [ ]:
def copy_json_files(
    data: pd.DataFrame,
    source_dir: Path,
    target_dir: Path,
    folder_col: str = "_source_folder",
    file_col: str = "_file_name",
    overwrite: bool = False,
) -> dict[str, int]:
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Source directory does not exist: {source_dir}")
    if folder_col not in data or file_col not in data:
        raise KeyError(f"Missing columns: {folder_col}, {file_col}")

    target_dir.mkdir(parents=True, exist_ok=True)
    summary = {"copied": 0, "skipped": 0, "missing": 0}

    files = data[[folder_col, file_col]].dropna().drop_duplicates()
    for folder, file_name in files.itertuples(index=False):
        source_file = source_dir / str(folder) / str(file_name)
        target_file = target_dir / str(file_name)

        if not source_file.is_file():
            summary["missing"] += 1
            print(f"Source file not found: {source_file}")
            continue
        if target_file.exists() and not overwrite:
            summary["skipped"] += 1
            continue

        shutil.copy2(source_file, target_file)
        summary["copied"] += 1

    return summary


In [ ]:
# sample_copy_summary = copy_folders(sample_df, SOURCE_DIR, SAMPLE_DIR)
# sample_copy_summary

In [ ]:
# sample_files_df = prepare_data(load_data(SAMPLE_DIR)) ## ich will die jsons aus dem sample (was schon gezogen wurde) kopieren 

# sample_json_copy_summary = copy_json_files(sample_files_df, SOURCE_DIR, JSON_DIR)
# sample_json_copy_summary